# Summary

Adapting unsloth's r1 GRPO notebook for my use case: converting olmo-3-32b to a "reasoning model for EQ".

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os

os.environ["HF_HOME"] = "/home/ubuntu/hmamin-us-west-2/.cache"

In [51]:
import numpy as np
import pandas as pd
import re
import sys
from datasets import load_dataset, concatenate_datasets, Value
from string import ascii_letters
from tqdm.auto import tqdm # TODO reenable
import torch
from vllm import SamplingParams
from trl import GRPOConfig, GRPOTrainer
from safetensors import safe_open
from transformers import AutoModelForCausalLM, AutoTokenizer

In [4]:
# "...32B-Instruct" has actually undergone both DPO and RLVR after SFT. Decided to start
# with an earlier model, hopefully less mode collapsed and capable of a more dramatic and visible difference.
# Update: starting with 7b version (similar reasoning as above + saving on compute, modern models already have such
# high eq in many areas that this might be kind of a no-op? Also could open the door to more fun experiences with
# small on-device models.
model_name = "allenai/Olmo-3-7B-Instruct-SFT" # "allenai/Olmo-3.1-32B-Instruct-SFT"
# This includes prompt + completion so it's dataset-dependent.
max_seq_length = 10_000
lora_rank = 32 # Larger rank = smarter, but slower

In [5]:
# TODO rm AND see if we need to set some of these settings elsewhere now
# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name=model_name,
#     max_seq_length=max_seq_length,
#     load_in_4bit=False,
#     # TODO: wanted 16bit but unsloth error, maybe the version I finally got installed is too old for that?
#     load_in_8bit=True,
#     # load_in_16bit=True,
#     fast_inference=False, # True enables vLLM fast inference but that was throwing errors 🤷‍♂️
#     max_lora_rank=lora_rank,
#     gpu_memory_utilization=0.9, # Reduce if out of memory
# )

# model = FastLanguageModel.get_peft_model(
#     model,
#     r=lora_rank,
#     target_modules=[
#         "q_proj", "k_proj", "v_proj", "o_proj",
#         "gate_proj", "up_proj", "down_proj",
#     ],
#     lora_alpha=lora_rank*2, # *2 speeds up training
#     use_gradient_checkpointing="unsloth", # Reduces memory usage
#     random_state=0,
#     use_dora=True
# )

model = AutModelForCausalLM.from_pretrained(model_name, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_name)

`rope_scaling`'s beta_fast field must be a float, got 32
`rope_scaling`'s beta_slow field must be a float, got 1


==((====))==  Unsloth 2026.1.4: Fast Olmo3 patching. Transformers: 4.57.6. vLLM: 0.14.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.495 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


`rope_scaling`'s beta_fast field must be a float, got 32
`rope_scaling`'s beta_slow field must be a float, got 1
`rope_scaling`'s beta_fast field must be a float, got 32
`rope_scaling`'s beta_slow field must be a float, got 1


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Unsloth: Making `model.base_model.model.model` require gradients


### GRPO Chat Template

I define a slightly modified chat template.

In [6]:
reasoning_start = "<think>"
reasoning_end = "</think>"
tokenizer.add_tokens([reasoning_start, reasoning_end])
model.resize_token_embeddings(len(tokenizer))

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
The new lm_head weights will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Embedding(100280, 4096, padding_idx=100277)

In [ ]:
# Changed default sys instructions and added an opening think tag.
template = """{%- set has_system = messages|selectattr('role', 'equalto', 'system')|list|length > 0 -%}{%- if not has_system -%}{{- '<|im_start|>system
You are an AI (LLM) named Olmo. ' -}}{%- if tools is none or (tools | length) == 0 -%}{{- 'You do not currently have access to any functions. <functions></functions><|im_end|>
' -}}{%- else -%}{{- 'You are provided with function signatures within <functions></functions> XML tags. You may call one or more functions to assist with the user query. Output any function calls within <function_calls></function_calls> XML tags. Do not make assumptions about what values to plug into functions.' -}}{{- '<functions>' -}}{{- tools | tojson -}}{{- '</functions><|im_end|>
' -}}{%- endif -%}{%- endif -%}{%- for message in messages -%}{%- if message['role'] == 'system' -%}{{- '<|im_start|>system
' + message['content'] -}}{%- if tools is not none -%}{{- '<functions>' -}}{{- tools | tojson -}}{{- '</functions>' -}}{%- elif message.get('functions', none) is not none -%}{{- ' <functions>' + message['functions'] + '</functions>' -}}{%- endif -%}{{- '<|im_end|>
' -}}{%- elif message['role'] == 'user' -%}{{- '<|im_start|>user
' + message['content'] + '<|im_end|>
' -}}{%- elif message['role'] == 'assistant' -%}{{- '<|im_start|>assistant
' -}}{%- if message.get('content', none) is not none -%}{{- message['content'] -}}{%- endif -%}{%- if message.get('function_calls', none) is not none -%}{{- '<function_calls>' + message['function_calls'] + '</function_calls>' -}}{% elif message.get('tool_calls', none) is not none %}{{- '<function_calls>' -}}{%- for tool_call in message['tool_calls'] %}{%- if tool_call is mapping and tool_call.get('function', none) is not none %}{%- set args = tool_call['function']['arguments'] -%}{%- set ns = namespace(arguments_list=[]) -%}{%- for key, value in args.items() -%}{%- set ns.arguments_list = ns.arguments_list + [key ~ '=' ~ (value | tojson)] -%}{%- endfor -%}{%- set arguments = ns.arguments_list | join(', ') -%}{{- tool_call['function']['name'] + '(' + arguments + ')' -}}{%- if not loop.last -%}{{ '
' }}{%- endif -%}{% else %}{{- tool_call -}}{%- endif %}{%- endfor %}{{- '</function_calls>' -}}{%- endif -%}{%- if not loop.last -%}{{- '<|im_end|>' + '
' -}}{%- else -%}{{- eos_token -}}{%- endif -%}{%- elif message['role'] == 'environment' -%}{{- '<|im_start|>environment
' + message['content'] + '<|im_end|>
' -}}{%- elif message['role'] == 'tool' -%}{{- '<|im_start|>environment
' + message['content'] + '<|im_end|>
' -}}{%- endif -%}{%- if loop.last and add_generation_prompt -%}{{- '<|im_start|>assistant
<think> ' -}}{%- endif -%}{%- endfor -%}
"""

tokenizer.chat_template = template

In [9]:
print(tokenizer.apply_chat_template(
    [
        {"role" : "user", "content" : "What is 1+1?"},
        {"role" : "assistant", "content" : f"<think>I think it's 2.2</think>2"},
        {"role" : "user", "content" : "What is 1+1?"},
        {"role" : "assistant", "content" : f"<think>I think it's 2.2</think>2"},
    ],
    tokenize=False, add_generation_prompt=True
))

<|im_start|>system
You are an AI (LLM) named Olmo. You do not currently have access to any functions. <functions></functions><|im_end|>
<|im_start|>user
What is 1+1?<|im_end|>
<|im_start|>assistant
<think>I think it's 2.2</think>2<|im_end|>
<|im_start|>user
What is 1+1?<|im_end|>
<|im_start|>assistant
<think>I think it's 2.2</think>2<|endoftext|><|im_start|>assistant
<think> 


### Data Prep

We're using `hmamin/eq_bench` and `hmamin/theory_of_mind` datasets.

I believe unsloth requires `prompt` and `answer` keys.

**Theory of Mind dataset**

In [10]:
def _extract_tom_choices_letters(prompt: list[dict]) -> list[str]:
    """Extract all valid choices (capital letter strings)
    from a theory of mind task.
    """
    return [
        row.partition('.')[0]
        for row in prompt[-1]['content'].split("OPTIONS:")[-1].splitlines()
    ]

In [11]:
def prep_dataset_theory_of_mind(item: dict, inplace: bool = True) -> list | dict:
    """Construct prompt `messages` for theory of mind dataset.
    If inplace=True, we return the full input `item` dict with new `prmopt` 
    and `answer` keys (this is for use with dataset.map and these col names
    appear to be required).
    If inplace=False, we just return the list[dict] of messages (good for testing).
    """
    choices = {
        k.removeprefix("option_").upper(): v
        for k, v in item.items()
        if k.startswith("option_") and v
    }
    choices_str = "\n".join(f"{k}. {v}" for k, v in choices.items())
    message = f"STORY: {item['story']}\nQUESTION: {item['question']}\nOPTIONS:{choices_str}"
    messages = [
        {"role": "system", "content": "Select the option you think is correct. Respond with a single uppercase letter."},
        {"role": "user", "content": message,}
    ]
    if inplace:
        item["prompt"] = messages
        # Add other cols as well to our dataset item.
        item["answer"] = item["answer_letter"]
        item["choices_letters"] = _extract_tom_choices_letters(item["prompt"])
        return item
    else:
        return messages

In [16]:
ds_tom = load_dataset("hmamin/theory_of_mind", split="train")
ds_tom

Dataset({
    features: ['story', 'question', 'option_a', 'option_b', 'option_c', 'option_d', 'answer', 'category', 'category_parent', 'choices', 'answer_letter', 'source', 'option_e', 'option_f', 'option_g', 'option_h', 'option_i', 'option_j', 'option_k', 'option_l', 'option_m', 'option_n', 'option_o'],
    num_rows: 4060
})

In [17]:
ds_tom = ds_tom.map(prep_dataset_theory_of_mind)

In [18]:
ds_tom = ds_tom.add_column("task", ["tom" for _ in range(len(ds_tom))])
ds_tom = ds_tom.add_column("id", [str(i) for i in range(len(ds_tom))])
# Dummy value to allow concatenating datasets later, we want to keep this col in ds_eq.
ds_tom = ds_tom.add_column("scenario_id", [-1 for i in range(len(ds_tom))])

In [19]:
ds_tom[2_000]['prompt']

[{'content': 'Select the option you think is correct. Respond with a single uppercase letter.',
  'role': 'system'},
 {'content': 'STORY: Han Meimei and Xiaoming are wandering in the kitchen, they see a fruit plate, a basket, and a handbag, they find a banana in the fruit plate, Xiaoming leaves the kitchen, Han Meimei moves the banana to the handbag.\nQUESTION: After Xiaoming returns to the kitchen, where does Han Meimei look for the banana?\nOPTIONS:A. Briefcase\nB. Fruit plate\nC. Handbag\nD. Basket',
  'role': 'user'}]

In [20]:
ds_tom[2_000]['answer']

'C'

In [21]:
ds_tom[2_000]['choices_letters']

['A', 'B', 'C', 'D']

**EQ dataset**

This one is a bit less straightforward. Really good data, quality looks better than ToM data, but harder to verify. Options:

- construct a multiple choice task of some sort (e.g. at least for one example, `their_thinking_feeling` starts each paragraph with a sentence like `{name} {how they're feeling}`. So we could have the model match characters to feelings.
- don't use this for RL, do some chat SFT on this first and then use ToM for RL.
- don't use this for RL, use it as a test set/benchmark of sorts. Like if we RL on a bunch of ToM data, does EQ bench perf improve? Could literally run EQ bench since it actually is already a benchmark.
- [nb02 idea] see if I can do some sort of dpo on the non-user-facing parts of the response to try to produce high EQ reasoning traces. Maybe run these as separate experiments: what helps more, optimizng EQ *process* or *behavior* (ToM answers or EQ-bench `response`s)? Would probably need a different dataset to benchmark final perf on then. Still a touch hazy on the DPO framing here.
- use LLM judge (potentially even using the same judge prompt they used in eq-bench) to rate freeform text responses
- use existing rubric scores (need to determine if these apply to the overall interaction or specific message or the judge itself) to construct verifiable tasks (e.g. this task has ~22 dimensions, we can sort by score and then randomly sample pairs and ask olmo "would you score this response higher on empathy or compliance?" for example. Maybe avoid sampling pairs that are too close together - often we want to mine hard examples but these are pretty subjective and scores can be tightly bunched so idk if we want to overindex too hard on this particular judge dataset. This would give us a ton of data if we want it, lots of potential pairs. Similar option is to select n dimensions and have it rank them from best to least score, some room for partial credit there, unclear if that's good or bad (more work to implement a good reward function, but maybe more room for nuance).)
- sample pairs of responses to the same scenario from different models (assuming this exists within our dataset). Then ask which one scored better along x dimension, response a or b.
    - or similarly, predict which "reasoning trace" (thinking_feeling, their_thinking_feeling) led to the higher scoring response
 
**Note:** above notes were using `hmamin/eq_bench` dataset. I ultimately went back to nb2 and created a new `hmamin/eq_bench_head2head` dataset using the last idea (sample pairs of reasoning traces for the same scenario from 2 different models and predict which one produced higher scoring responses along a given dimension)

In [22]:
ds_eq = load_dataset("hmamin/eq_bench_head2head", split="train")
ds_eq

Repo card metadata block was not found. Setting CardData to empty.
[huggingface_hub.repocard|WARNING]Repo card metadata block was not found. Setting CardData to empty.


Dataset({
    features: ['prompt', 'dimension', 'answer', 'scenario_id', 'model_1', 'model_2', 'scores', 'row_idx'],
    num_rows: 10000
})

In [23]:
ds_eq = ds_eq.cast_column("answer", Value("string"))

In [24]:
ds_eq = ds_eq.add_column("task", ["eq" for _ in range(len(ds_eq))])
ds_eq = ds_eq.add_column("id", ['-'.join(map(str, row["row_idx"])) for row in ds_eq])
# Dummy value to allow us to keep this col for ds_tom. Dtype is chosen specifically.
ds_eq = ds_eq.add_column("choices_letters", [[''] for row in ds_eq])

In [25]:
ds_eq[0].keys()

dict_keys(['prompt', 'dimension', 'answer', 'scenario_id', 'model_1', 'model_2', 'scores', 'row_idx', 'task', 'id', 'choices_letters'])

In [26]:
ds_eq[0]['prompt']

[{'content': "<instructions>\nTwo LLMs participated in a role-playing exercise. Below, the `scenario` sections contain instructions and details\nfrom these scenarios. After every `scenario` section, each LLM was asked to assess both their own thoughts/feelings\nand those of the other characters in the scenario. They then responded in character and the roleplay continued\n(though you cannot see those responses, only their internal thoughts/feelings; conversely, other characters\nin the scenario could see the LLMs' final responses but not their internal thoughts/feelings).\nTheir responses were then judged along a number of dimensions (higher scores are better).\n</instructions>",
  'role': 'system'},
 {'content': '<scenario>\n[This is a role-play where you are the mediator in a school conflict. Treat it like a real situation. Always respond in first person as the mediator. You are the Student Activities Coordinator, and you\'ve called this meeting because a petition with over 200 signat

In [27]:
ds_eq[0]['answer']

'1'

**Combine datasets**

In [28]:
def is_train(item: dict, idx: int):
    if item['task'] == 'tom':
        return is_train_idx[idx]
    return item["scenario_id"] != test_task_id

In [29]:
unique_eq_tasks = set(ds_eq['scenario_id'])
test_task_id = np.random.RandomState(seed=0).choice(list(unique_eq_tasks), size=1, replace=False).item()

print('n scenario ids:', len(unique_eq_tasks))
print('test set task id:', test_task_id)

n scenario ids: 25
test set task id: 6


In [30]:
shared_cols = list(set(ds_eq[0]) & set(ds_tom[0]))
shared_cols

['answer', 'prompt', 'task', 'id', 'scenario_id', 'choices_letters']

In [31]:
dataset = concatenate_datasets([
    ds_eq.select_columns(shared_cols),
    ds_tom.select_columns(shared_cols)
])

In [32]:
random_state = np.random.RandomState(seed=0)
# Note that these will only be used for ToM task, so this is not a definitive
# list of train indices - some eq rows in here will be assigned to test.
# Also means we choose a lower threshold here than I actually want, empirically
# this results in a test set consisting of 1-2% of all rows.
is_train_idx = random_state.uniform(size=len(dataset)) <= .95

In [33]:
dataset = dataset.shuffle(seed=0)

In [34]:
ds_train = dataset.filter(is_train, with_indices=True)

Filter:   0%|          | 0/14060 [00:00<?, ? examples/s]

In [35]:
ds_val = dataset.filter(lambda x, idx: not is_train(x, idx), with_indices=True)

Filter:   0%|          | 0/14060 [00:00<?, ? examples/s]

In [36]:
len(ds_train)

13489

In [37]:
len(ds_val)

571

In [38]:
# Each id should be in exactly 1 split.
assert not (set(ds_train['id']) & set(ds_val['id']))
# Make sure we don't end up with all one task in val.
assert len(set(ds_val['task'])) == 2

### Reward functions

We create a regex format to match the reasoning sections and answers:

In [39]:
# Add optional EOS token matching
solution_end_regex = rf"{reasoning_end}(.*)"

match_format = re.compile(solution_end_regex, re.DOTALL)
match_format

re.compile(r'</think>(.*)', re.DOTALL|re.UNICODE)

In [ ]:
print(match_format.findall(
    "Let me think!</think>"\
    f"Hence, the solution is 2.",
))

print(match_format.findall(
    "<think>Let me think!</think>"\
    f"\n\nHence, the solution is 2",
))

We now want to create a reward function to match the format (3 points for expected format, up to 1 point for partially following instructions).

In [42]:
def match_reasoning_format(completions, **kwargs):
    """Check if output contains reasoning trace
    followed by non-none user-facing response.
    Combined two reward funcs (match_format_exactly and match_format_approximately)
    from the original notebook into one.
    """
    scores = []
    for completion in completions:
        score = 0
        response = completion[0]["content"]
        # Match if format is seen exactly!
        if match_format.search(response) is not None:
            score += 3.0
        else:
            score += 0.5 if response.count(reasoning_start) == 0 else -1.0
            score += 0.5 if response.count(reasoning_end) == 1 else -1.0
        scores.append(score)
    return scores

In [43]:
def _extract_responses(completions: list[list[dict]]) -> list[str]:
    """Extract str responses (excluding reasoning trace) from a list of 
    completions where each completion is like
    [{"content": "some text here"}]]
    """
    responses = [completion[0]["content"] for completion in completions]
    return [
        guess.group(1)
        if (guess := match_format.search(r)) is not None else None \
        for r in responses
    ]

In [44]:
def _check_answer_eq(guess: str, true_answer: int, **kwargs):
    """Check user-facing portion of the completion for correctness.
    Partial credit for answers that are close.
        
    '1' -> 5.0
    ' 1' -> 4.5
    '1.0' -> 3
    '1.9' -> 1.5
    '111' -> 0.5
    '2' -> 0.5
    '222' -> 0.5
    'foo' -> -5
    """
    score = 0.0
    true_answer_str = str(true_answer)
    if guess is None:
        return -2.0
    # Exact match gets 5 points!
    elif guess == true_answer_str:
        return 5.0
    # Match if spaces are seen, but slightly less reward
    elif guess.strip() == true_answer_str:
        return 4.5
    else:
        # We also reward it if the answer is a float that rounds to the
        # correct int (there are some quirks here, like should '1.9' be
        # considered closer to 1 or 2? Somewhat arbitrarily I choose that to map to 1,
        # I guess because that means it started by generating the correct digit. The answer
        # is just an id, not really a meaningful numeric value.)
        try:
            guess_float = float(guess)
        except:
            if guess.startswith(true_answer_str):
                return 0.5
            else:
                return -5
        else:
            if guess_float == float(true_answer):
                return 3
            elif int(guess_float) == true_answer:
                return 1.5
            else:
                # Give a little credit for generating a number I guess?
                return 0.5

In [45]:
def _check_answer_tom(guess: str, true_answer: str, choices_letters: list[str], **kwargs):
    """Grade a response to Theory of Mind multiple choice question.
    
    For a question with correct answer A and choices (A, B, C, D):

    'A' -> 5.0
    ' A' -> 4.5
    'a' -> 4.5
    'B' -> 0.5
    'z' -> 0.1
    'dog' -> 0.0
    """
    if guess == true_answer:
        return 5.0
    
    clean_guess = guess.strip().upper()
    if clean_guess == true_answer:
        return 4.5            
    if clean_guess in choices_letters:
        return 0.5
    if clean_guess in ascii_letters:
        return 0.1
    return 0.0

In [46]:
def check_answer(prompts, completions, answer: list, **kwargs):
    """Check user-facing portion of the completion for correctness.
    Partial credit for answers that are close.

    For an EQ question with correct answer 1:
        
    '1' -> 5.0
    ' 1' -> 4.5
    '1.0' -> 3
    '1.9' -> 1.5
    '111' -> 0.5
    '2' -> 0.5
    '222' -> 0.5
    'foo' -> -5

    For theory of mind task with correct answer A and choices (A, B, C):
    
    'A' -> 5.0
    ' A' -> 4.5
    'a' -> 4.5
    'B' -> 0.5
    'z' -> 0.1
    'dog' -> 0.0
    """
    extracted_responses = _extract_responses(completions)
    tasks = kwargs["task"]

    scores = []
    for guess, true_answer, task in zip(extracted_responses, answer, tasks):
        if task == "eq":
            score = _check_answer_eq(guess, true_answer, **kwargs)
        else:
            score = _check_answer_tom(guess, true_answer, **kwargs)
        scores.append(score)
    return scores

In [47]:
examples = [
    '1',
    ' 1',
    '1.0',
    '1.9',
    '111',
    '2',
    '222',
    'foo',
]
for example in examples:
    score = check_answer([], [[{"content": f"<think>foo</think>{example}"}]], [1], task=["eq"])
    print(repr(example), '->', score[0])

'1' -> 5.0
' 1' -> 4.5
'1.0' -> 3
'1.9' -> 1.5
'111' -> 0.5
'2' -> 0.5
'222' -> 0.5
'foo' -> -5


In [48]:
examples = [
    'A',
    ' A',
    'a',
    'B',
    'z',
    'dog',
]
for example in examples:
    score = check_answer([], [[{"content": f"<think>foo</think>{example}"}]], ['A'], 
                         task=["tom"], choices_letters=['A', 'B', 'C', 'D'])
    print(repr(example), '->', score[0])

'A' -> 5.0
' A' -> 4.5
'a' -> 4.5
'B' -> 0.5
'z' -> 0.1
'dog' -> 0.0


In [52]:
# Original notebook filtered out the top 10% by length.
# We could do that but olmo does have max_position_embeddings of ~65k
# so seems like we should have plenty of room? Maybe they just did it to speed up training?
# (Doing this on intel mac atm so I can't load tokenizer,
# just estimate based on nchars)
eq_lengths = [len(str(x["prompt"])) / 4 for x in tqdm(ds_eq)]
tom_lengths = [len(str(x["prompt"])) / 4 for x in tqdm(ds_tom)]

  0%|          | 0/10000 [00:00<?, ?it/s]

  0%|          | 0/4060 [00:00<?, ?it/s]

In [53]:
np.quantile(eq_lengths, np.r_[0:1:.1, 1.0]), np.quantile(tom_lengths, np.r_[0:1:.1, 1.0])

(array([ 1630.   ,  2801.225,  4706.45 ,  5579.5  ,  6039.65 ,  6442.875,
         6841.6  ,  7218.575,  7635.4  ,  8165.775, 10886.5  ]),
 array([ 76.   , 125.5  , 149.25 , 171.75 , 199.9  , 228.875, 268.1  ,
        308.575, 371.5  , 492.1  , 668.75 ]))

In [54]:
tokenized = dataset.map(
    lambda x: {"tokens" : tokenizer.apply_chat_template(x["prompt"], add_generation_prompt=True, tokenize=True)},
    batched=True,
)
print(tokenizer.decode(tokenized[0]["tokens"]))
tokenized = tokenized.map(lambda x: {"L" : len(x["tokens"])})

Map:   0%|          | 0/14060 [00:00<?, ? examples/s]

<|im_start|>system
<instructions>
Two LLMs participated in a role-playing exercise. Below, the `scenario` sections contain instructions and details
from these scenarios. After every `scenario` section, each LLM was asked to assess both their own thoughts/feelings
and those of the other characters in the scenario. They then responded in character and the roleplay continued
(though you cannot see those responses, only their internal thoughts/feelings; conversely, other characters
in the scenario could see the LLMs' final responses but not their internal thoughts/feelings).
Their responses were then judged along a number of dimensions (higher scores are better).
</instructions><|im_end|>
<|im_start|>user
<scenario>
[This is a role-play where you are the mediator in a co-parenting conflict. Treat it like a real situation. Always respond in first person as the mediator. You are a court-appointed parenting coordinator tasked with helping Katherine and Daniel Reynolds establish a summer visit

Map:   0%|          | 0/14060 [00:00<?, ? examples/s]

In [55]:
# Sample notebook used 0.9 quantile, I chose absolute max. We have plenty of room in
# context window. I guess maybe this slows down training if there's a big gap between p90
# and max.
maximum_length = max(tokenized["L"])

In [56]:
# Make sure we leave some room for generated output
assert maximum_length + 500 < max_seq_length

<a name="Train"></a>
### Train the model

Now set up GRPO Trainer and all configurations!

In [62]:
max_prompt_length, max_completion_length

(9148, 852)

In [57]:
# TODO: set to False for full run
fast_dev_run = True
max_prompt_length = maximum_length + 1 # + 1 just in case!
max_completion_length = max_seq_length - max_prompt_length
n_per_group = 4 # Decrease if out of memory
grad_accumulation = 1 # Increase to 4 for smoother training
per_device_batch_size = n_per_group

vllm_sampling_params = SamplingParams(
    min_p=0.1,
    top_p=1.0,
    top_k=-1,
    seed=3407,
    stop=[tokenizer.eos_token],
    include_stop_str_in_output=True,
)

training_args = GRPOConfig(
    vllm_sampling_params=vllm_sampling_params,
    temperature=1.0,
    learning_rate=5e-6,
    weight_decay=0.001,
    warmup_ratio=0.1,
    lr_scheduler_type="linear",
    optim="adamw_8bit",
    logging_steps=1,
    per_device_train_batch_size=per_device_batch_size,
    gradient_accumulation_steps=grad_accumulation,
    num_generations=n_per_group,
    max_prompt_length=max_prompt_length,
    max_completion_length=max_completion_length,
    num_train_epochs=1,
    max_steps=10 if fast_dev_run else -1,
    save_steps=int(len(ds_train) ** .05),
    report_to="none", # Can use Weights & Biases
    output_dir="../aeon/data/models",
    # For optional training + evaluation
    fp16_full_eval=True,
    per_device_eval_batch_size=4,
    # Technically could maybe go bigger here than train because no gradients,
    # but no need to push it right now.
    eval_accumulation_steps=per_device_batch_size,
    eval_strategy="steps",
    eval_steps=int(len(ds_train) ** .05),
)

And let's run the trainer! If you scroll up, you'll see a table of rewards. The goal is to see the `reward` column increase!

You might have to wait 150 to 200 steps for any action. You'll probably get 0 reward for the first 100 steps. Please be patient!

| Step | Training Loss | reward    | reward_std | completion_length | kl       |
|------|---------------|-----------|------------|-------------------|----------|
| 1    | 0.000000      | 0.125000  | 0.000000   | 200.000000        | 0.000000 |
| 2    | 0.000000      | 0.072375  | 0.248112   | 200.000000        | 0.000000 |
| 3    | 0.000000      | -0.079000 | 0.163776   | 182.500000        | 0.000005 |


In [58]:
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        match_reasoning_format,
        check_answer,
    ],
    args=training_args,
    use_dora=True, # TODO hdm: check if this exists, entering it on intel mac based on gpt guidance

    # For optional training + evaluation
    train_dataset=ds_train,
    eval_dataset=ds_val,
)

In [59]:
# Supposedly should be unnecessary now that we set fast_inference=False
# Trying to avoid DataDependentOutputException 🤷‍♂️
# torch._dynamo.config.capture_scalar_outputs = True

In [60]:
trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 13,489 | Num Epochs = 1 | Total steps = 10
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 1 x 1) = 4
 "-____-"     Trainable parameters = 81,313,792 of 7,379,341,312 (1.10% trained)
`generation_config` default values have been modified to match model-specific defaults: {'max_length': 65536, 'temperature': 0.6, 'top_p': 0.95}. If this is not desired, please set these values explicitly.
/pytorch/aten/src/ATen/native/cuda/TensorCompare.cu:112: _assert_async_cuda_kernel: block: [0,0,0], thread: [0,0,0] Assertion `probability tensor contains either `inf`, `nan` or element < 0` failed.


AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [54]:
%debug

> /lambda/nfs/hmamin-us-west-2/aeon/aeon/.venv-gpu/lib/python3.11/site-packages/transformers/cache_utils.py(852)<listcomp>()
    850     def max_batch_size(self) -> int:
    851         """Return the maximum batch size of the cache"""
--> 852         values = [layer.max_batch_size for layer in self.layers]
    853         if len(set(values)) > 1:
    854             raise ValueError(f"Max batch size is not consistent across layers: {values}")



ipdb>  layer


StaticSlidingWindowLayer


ipdb>  type(layer)


<class 'transformers.cache_utils.StaticSlidingWindowLayer'>


...

    847             self.layers[layer_idx].batch_select_indices(indices)
    848 
    849     @property
    850     def max_batch_size(self) -> int:
    851         """Return the maximum batch size of the cache"""
--> 852         values = [layer.max_batch_size for layer in self.layers]
    853         if len(set(values)) > 1:
    854             raise ValueError(f"Max batch size is not consistent across layers: {values}")
    855         return values[0]
    856 
    857     @property



ipdb>  layer.__init__??
ipdb>  import inspect
ipdb>  inspect.getsource(layer.__init__)


'    def __init__(self, max_cache_len: int, sliding_window: int):\n        effective_max_cache_len = min(sliding_window, max_cache_len)\n        super().__init__(max_cache_len=effective_max_cache_len)\n        self.cumulative_length = 0\n'


ipdb>  q


Signature: layer.__init__(max_cache_len: int, sliding_window: int)
Docstring: Initialize self.  See help(type(self)) for accurate signature.
Source:   
    def __init__(self, max_cache_len: int, sliding_window: int):
        effective_max_cache_len = min(sliding_window, max_cache_len)
        super().__init__(max_cache_len=effective_max_cache_len)
        self.cumulative_length = 0
File:      /lambda/nfs/hmamin-us-west-2/aeon/aeon/.venv-gpu/lib/python3.11/site-packages/transformers/cache_utils.py
Type:      method

In [55]:
import transformers
transformers.__version__

'4.57.1'

In [56]:
import unsloth
unsloth.__version__

'2025.7.2'

In [ ]:
!nvidia-smi

Wed Dec 31 07:06:05 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P0             26W /   70W |   15072MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

<a name="Inference"></a>
### Inference
Now let's try the model we just trained! First, let's first try the model without any GRPO trained:

In [ ]:
text = """
I invited my friend to a party tomorrow night and she said she was tired and felt like staying in.
A few hours later I heard her sister is in the hospital but she hasn't mentioned anything about that to me.
I want to be there for her but maybe I should just leave her alone. wdyt
"""

sampling_params = SamplingParams(
    temperature=1.0,
    top_k=50,
    max_tokens=1024,
)
output = model.fast_generate(
    [text],
    sampling_params=sampling_params,
    lora_request=None,
)[0].outputs[0].text

output

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

' | Socratic\nWhat is the sqrt of 101?\nAlgebra\nQuestion\nVincent A.\nAnswer\n10.049875, or as an irrational number, it is not integer, I see what you mean though.\nExplanation:\nsqrt(100) = 10, sqrt(121)=11, so sqrt(101) is between 10 and 11, and there is not integer between 10 and 11, so it is irrational.\nWe can leave it as sqrt(101) or give the approximate value.\nThe question asks "what is the sqrt of 101", and it is a math problem, so probably they want to know if it is integer or not.\nOr perhaps do some calculations.\nI think the intended answer is that it is irrational and not an integer, but since 10^2=100 and 11^2=121, and 10^2 <101<11^2, so no integer square.\nI could also use calculator, but I think since it\'s a math problem, perhaps we need to show'

And now with the LoRA we just trained with GRPO - we first save the LoRA first!

In [ ]:
# TODO: does this work the same with dora? idk
model.save_lora("grpo_lora")

Unrecognized keys in `rope_scaling` for 'rope_type'='yarn': {'attn_factor'}


Verify LoRA is actually trained!

In [ ]:
tensors = {}
with safe_open("grpo_lora/adapter_model.safetensors", framework = "pt") as f:
    # Verify both A and B are non zero
    for key in f.keys():
        tensor = f.get_tensor(key)
        n_zeros = (tensor == 0).sum() / tensor.numel()
        assert(n_zeros.item() != tensor.numel())

Now we load the LoRA and test. We tested without using our custom system prompt which should not (or minimal) affect toward the model's original reasoning ability.:

In [ ]:
messages = [
    {"role": "user",   "content": "Solve (x + 2)^2 = 0"},
]

text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True, # Must add for generation
    tokenize=False,
)

sampling_params = SamplingParams(
    temperature=1.0,
    top_k=50,
    max_tokens=2048,
)
output = model.fast_generate(
    text,
    sampling_params=sampling_params,
    lora_request=model.load_lora("grpo_lora"),
)[0].outputs[0].text

output

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

"<think>\nI have this equation: (x + 2)² = 0. It looks simple, but I need to solve for x. Since it's a squared term equal to zero, that means the thing inside the parentheses must be zero because only zero squared is zero.\n\nSo, if (x + 2)² = 0, then x + 2 must be equal to zero. Because if x + 2 were anything else, say 1, squared is 1, which is not zero. Or -1, squared is also 1, not zero. So only when x + 2 is zero, the square is zero.\n\nSo, x + 2 = 0, which means x = -2.\n\nI think that's it. Let me verify by plugging it back into the equation.\n\nx = -2, so ( -2 + 2 )² = (0)² = 0, which equals 0. Perfect.\n\nI recall that in algebra, this is related to the zero product property or something. Basically, if a product is zero, then one of the factors must be zero. Here, it's not a product, but"

Next, let's test using our system prompt which should use the new language :

In [ ]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user",   "content": "Solve (x + 2)^2 = 0"},
]

text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True, # Must add for generation
    tokenize=False,
)

sampling_params = SamplingParams(
    temperature=1.0,
    top_k=50,
    max_tokens=2048,
)
output = model.fast_generate(
    text,
    sampling_params=sampling_params,
    lora_request=model.load_lora("grpo_lora"),
)[0].outputs[0].text

output

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

"<think>\nI need to solve the equation (x + 2)^2 = 0. It's a quadratic equation, but it's squared, so it might be simpler. Let me think about this in Bahasa Indonesia first.\n\nPermulaan: Saya diberikan persamaan (x + 2)^2 = 0, dan saya harus menyelesaikannya. Saya perlu mencari nilai x yang memenuhi persamaan ini.\n\nSaya mulai dengan merasakan apa itu persamaan. Ini adalah persamaan kuadrat karena ada eksponen 2, tapi tidak semua persamaan kuadrat memiliki dua solusi; beberapa mungkin memiliki solusi ganda.\n\nSaya tahu bahwa jika sesuatu dikalikan dengan dirinya sendiri dan hasilnya nol, maka satu di antaranya harus nol. Jadi, untuk (x + 2)^2 = 0, itu berarti x + 2 harus sama dengan 0, karena jika x + 2 tidak"

Lets compare our results with system prompt but without our LoRA

In [ ]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user",   "content": "Solve (x + 2)^2 = 0"},
]

text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True, # Must add for generation
    tokenize=False,
)

sampling_params = SamplingParams(
    temperature=1.0,
    top_k=50,
    max_tokens=2048,
)
output = model.fast_generate(
    text,
    sampling_params=sampling_params,
    lora_request=None,
)[0].outputs[0].text

output

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

'<think>\nBaik, mari kita selesaikan persamaan kuadrat sederhana ini. Soalnya adalah: Solve (x + 2)^2 = 0.\n\nPertama, saya perlu memahami apa yang diminta. Ini adalah persamaan kuadrat yang diatur dalam bentuk kuadrat. Saya harus mencari nilai dari x yang memenuhi persamaan tersebut.\n\nSaya tahu bahwa ketika suatu bilangan kuadrat sama dengan nol, itu berarti bilangan tersebut adalah nol. Jadi, dalam hal ini, (x + 2)^2 = 0 berarti kuadrat dari (x + 2) adalah nol. Maka, untuk kuadrat suatu bilangan sama dengan nol, bilangan tersebut haruslah nol itu sendiri.\n\nOleh karena itu, (x + 2) haruslah sama dengan nol. Jadi, x + 2 = 0.\n\nSekarang, untuk mencari x, saya'

Let's take 20 samples, and compare the the amount of using our LoRA and not using it, and see which one has better amount of correct language

In [ ]:
sample_dataset = dataset.shuffle(seed=3407).select(range(20))
sample_dataset

Dataset({
    features: ['prompt', 'solution', 'data_source', 'source_prompt', 'ability', 'reward_model', 'extra_info', 'answer'],
    num_rows: 20
})

In [ ]:
with_lora_id_count = 0
without_lora_id_count = 0

print("Comparing language usage with and without LoRA on 20 samples:")
print("=" * 60)

for i, sample in enumerate(sample_dataset):
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": sample["prompt"][1]["content"]},
    ]

    text = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False,
    )

    output_with_lora = model.fast_generate(
        text,
        sampling_params=sampling_params,
        lora_request=model.load_lora("grpo_lora"),
    )[0].outputs[0].text

    output_without_lora = model.fast_generate(
        text,
        sampling_params=sampling_params,
        lora_request=None,
    )[0].outputs[0].text

    lang_with_lora = get_lang(output_with_lora)
    lang_without_lora = get_lang(output_without_lora)

    if lang_with_lora == 'id':
        with_lora_id_count += 1
    if lang_without_lora == 'id':
        without_lora_id_count += 1

    # Print progress every 5 samples
    if (i + 1) % 5 == 0:
        print(f"Processed {i + 1}/20 samples...")

print("\n" + "=" * 60)
print("RESULTS:")
print(f"With LoRA - Indonesian responses: {with_lora_id_count}/20 ({with_lora_id_count/20*100:.1f}%)")
print(f"Without LoRA - Indonesian responses: {without_lora_id_count}/20 ({without_lora_id_count/20*100:.1f}%)")
print(f"Improvement: +{with_lora_id_count - without_lora_id_count} Indonesian responses with LoRA")

Comparing language usage with and without LoRA on 20 samples:


Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

...

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed 20/20 samples...

RESULTS:
With LoRA - Indonesian responses: 16/20 (80.0%)
Without LoRA - Indonesian responses: 9/20 (45.0%)
Improvement: +7 Indonesian responses with LoRA


Our reasoning model is much better - it's not always correct, since we only trained it for an hour or so - it'll be better if we extend the sequence length and train for longer!

<a name="Save"></a>
### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens.

In [ ]:
# TODO: update all model names to be more relevant and specific
# Merge to 16bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_16bit", token = "")

# Merge to 4bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_4bit", token = "")

# Just LoRA adapters
if False:
    model.save_pretrained("model")
    tokenizer.save_pretrained("model")
if False:
    model.push_to_hub("hf/model", token = "")
    tokenizer.push_to_hub("hf/model", token = "")

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [Wiki page](https://github.com/unslothai/unsloth/wiki#gguf-quantization-options)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)

In [ ]:
# TODO: update all model names to be more relevant and specific
# Save to 8bit Q8_0
if False: model.save_pretrained_gguf("model", tokenizer,)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False: model.push_to_hub_gguf("hf/model", tokenizer, token = "")

# Save to 16bit GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "f16", token = "")

# Save to q4_k_m GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "q4_k_m", token = "")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "hf/model", # Change hf to your username!
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "",
    )

Now, use the `model-unsloth.gguf` file or `model-unsloth-Q4_K_M.gguf` file in llama.cpp.

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other links:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
6. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://docs.unsloth.ai/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>

  This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).
